# Readmission Risk Classifier
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Hilina Fissha Woreta**

This notebook trains the XGBoost readmission risk classifier on the preprocessed MIMIC-IV data. The model takes patient demographics, comorbidities, lab values, ICU features, and prior admissions as input and outputs a probability of 30-day readmission. We then calibrate those probabilities and use them to drive the Stage 3 enrollment decision.

## Setup and data loading

Loading the three splits from the preprocessing notebook output. The test set stays untouched throughout this notebook.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

random_seed = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

data_dir = "/kaggle/input/notebooks/hilinafissha16/preprocessing"

train = pd.read_parquet(f"{data_dir}/splits/train.parquet")
val   = pd.read_parquet(f"{data_dir}/splits/val.parquet")
test  = pd.read_parquet(f"{data_dir}/splits/test.parquet")

print("train:", train.shape)
print("val:  ", val.shape)
print("test: ", test.shape)

train: (279558, 43)
val:   (59905, 43)
test:  (59906, 43)


## Feature preparation

We drop columns that shouldn't be model inputs: identifiers, dates, the target variable, and the raw string versions of categorical columns. The categorical columns get replaced with numeric encodings since XGBoost only works with numbers.

We also drop `hospital_expire_flag` specifically because it leaks information. A patient who died in hospital can't be readmitted, so the model would learn to use death as a shortcut rather than learning actual readmission patterns.

In [ ]:
cols_to_drop = [
    "subject_id", "hadm_id", "admittime", "dischtime",
    "readmitted_30d", "race_x_sex", "race_x_insurance",
    "race_clean", "insurance_clean", "sex", "admission_type",
    "hospital_expire_flag"
]

for df in [train, val, test]:
    df["sex_enc"]            = df["sex"].map({"Male": 0, "Female": 1}).fillna(-1)
    df["race_enc"]           = pd.Categorical(df["race_clean"]).codes
    df["insurance_enc"]      = pd.Categorical(df["insurance_clean"]).codes
    df["admission_type_enc"] = pd.Categorical(df["admission_type"]).codes

feature_cols = [c for c in train.columns if c not in cols_to_drop]
feature_cols = list(dict.fromkeys(feature_cols))

target = "readmitted_30d"

x_train = train[feature_cols]
y_train = train[target]
x_val   = val[feature_cols]
y_val   = val[target]
x_test  = test[feature_cols]
y_test  = test[target]

print("number of features:", len(feature_cols))
print("feature list:", feature_cols)
print("\nclass balance in train:")
print(y_train.value_counts())
print("\nreadmission rate:", round(y_train.mean() * 100, 1), "%")

number of features: 35
feature list: ['age', 'los_days', 'cm_chf', 'cm_arrhythmia', 'cm_hypertension', 'cm_cpd', 'cm_diabetes', 'cm_renal_failure', 'cm_liver_disease', 'cm_cancer', 'cm_obesity', 'cm_depression', 'cm_anxiety', 'cm_alcohol_abuse', 'cm_drug_abuse', 'cm_psychosis', 'cm_coagulopathy', 'cm_aids', 'comorbidity_count', 'bicarbonate', 'bun', 'creatinine', 'glucose', 'hemoglobin', 'platelets', 'potassium', 'sodium', 'wbc', 'icu_los_total', 'n_icu_stays', 'prior_admissions_12m', 'sex_enc', 'race_enc', 'insurance_enc', 'admission_type_enc']

class balance in train:
readmitted_30d
0    220411
1     59147
Name: count, dtype: int64

readmission rate: 21.2 %


## Baseline model

Training a first version with sensible default parameters just to confirm everything works end to end and get a baseline AUROC before tuning.

`scale_pos_weight` handles the class imbalance, since about 79% of patients were not readmitted and 21% were. Without this the model would just predict 0 for everyone.

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=1
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50
)

val_preds = model.predict_proba(x_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"\nbaseline validation AUROC: {val_auc:.4f}")

[0]	validation_0-auc:0.61722
[50]	validation_0-auc:0.69233
[100]	validation_0-auc:0.69589
[150]	validation_0-auc:0.69749
[200]	validation_0-auc:0.69817
[250]	validation_0-auc:0.69879
[299]	validation_0-auc:0.69917

baseline validation AUROC: 0.6992


## Feature importance

Looking at which features the model relied on most. This is important for the fairness analysis because if protected attributes like race or insurance type show up high on this list, it means the model is using them directly to make predictions.

In [ ]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("top 15 most important features:")
print(importance.head(15).to_string(index=False))

top 15 most important features:
             feature  importance
prior_admissions_12m    0.335626
         n_icu_stays    0.045568
            race_enc    0.039689
          hemoglobin    0.038971
  admission_type_enc    0.037451
            los_days    0.035893
           cm_cancer    0.026775
        cm_psychosis    0.023610
             sex_enc    0.023562
       icu_los_total    0.023011
       insurance_enc    0.020708
              sodium    0.020455
                 wbc    0.019349
     cm_hypertension    0.019046
           platelets    0.018537


## Hyperparameter tuning

Using Optuna to search for better hyperparameters. We run 30 trials and pick the combination that gives the highest AUROC on the validation set.

The model learns only from the training set during this process. The validation set is used purely to score each trial.

In [ ]:
def objective(trial):
    params = {
        "n_estimators":        trial.suggest_int("n_estimators", 200, 600),
        "learning_rate":       trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth":           trial.suggest_int("max_depth", 4, 8),
        "subsample":           trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight":    trial.suggest_int("min_child_weight", 1, 10),
        "scale_pos_weight":    (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state":        random_seed,
        "eval_metric":         "auc",
        "early_stopping_rounds": 20,
    }

    m = xgb.XGBClassifier(**params, verbosity=0)
    m.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    preds = m.predict_proba(x_val)[:, 1]
    return roc_auc_score(y_val, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("best AUROC:", round(study.best_value, 4))
print("best params:", study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

best AUROC: 0.7006
best params: {'n_estimators': 507, 'learning_rate': 0.02353108346740046, 'max_depth': 8, 'subsample': 0.6314578878856523, 'colsample_bytree': 0.6154552406711897, 'min_child_weight': 5}


## Final model

Retraining with the best parameters found by Optuna. This is the model we carry forward into the fairness experiments.

In [ ]:
best_params = study.best_params

best_model = xgb.XGBClassifier(
    **best_params,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=random_seed,
    eval_metric="auc",
    early_stopping_rounds=20,
    verbosity=1
)

best_model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50
)

val_preds = best_model.predict_proba(x_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f"\nfinal validation AUROC: {val_auc:.4f}")

[0]	validation_0-auc:0.61857
[50]	validation_0-auc:0.69445
[100]	validation_0-auc:0.69644
[150]	validation_0-auc:0.69783
[200]	validation_0-auc:0.69890
[250]	validation_0-auc:0.69965
[300]	validation_0-auc:0.70005
[350]	validation_0-auc:0.70042
[400]	validation_0-auc:0.70059
[423]	validation_0-auc:0.70060

final validation AUROC: 0.7006


## Calibration

Raw XGBoost scores are not well-calibrated probabilities. For example before calibration the model was outputting a mean predicted probability of 45% when the actual readmission rate is only 21%. Isotonic regression fixes this by mapping the raw scores to calibrated probabilities that match the actual observed rates.

This matters for Stage 3 because we're using these scores to make enrollment decisions, and the threshold we set need to reflect real probabilities.

In [ ]:
iso_reg = IsotonicRegression(out_of_bounds="clip")
iso_reg.fit(val_preds, y_val)

val_preds_calibrated = iso_reg.predict(val_preds)

print("before calibration:")
print("  mean predicted probability:", round(val_preds.mean(), 3))
print("  actual readmission rate:   ", round(y_val.mean(), 3))

print("\nafter calibration:")
print("  mean predicted probability:", round(val_preds_calibrated.mean(), 3))
print("  actual readmission rate:   ", round(y_val.mean(), 3))

before calibration:
  mean predicted probability: 0.457
  actual readmission rate:    0.212

after calibration:
  mean predicted probability: 0.212
  actual readmission rate:    0.212


## Saving the model and predictions

Saving the trained model and calibrator so Stage 3 and the fairness experiments can load them without retraining.

In [ ]:
with open("/kaggle/working/stage2_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("/kaggle/working/stage2_calibrator.pkl", "wb") as f:
    pickle.dump(iso_reg, f)

val_preds_df = pd.DataFrame({
    "hadm_id": val["hadm_id"].values,
    "y_true":  y_val.values,
    "y_pred":  val_preds,
    "y_pred_calibrated": val_preds_calibrated
})

val_preds_df.to_parquet("/kaggle/working/val_predictions.parquet", index=False)

print("saved:")
print("  stage2_model.pkl")
print("  stage2_calibrator.pkl")
print("  val_predictions.parquet")

saved:
  stage2_model.pkl
  stage2_calibrator.pkl
  val_predictions.parquet


## Care management enrollment

This stage applies a threshold to the calibrated risk scores and enrolls the top 10% of patients into the care management program. This directly mirrors the setting from Obermeyer et al. 2019.

We then look at enrollment rates broken down by race. If the rates differ significantly across groups, that's evidence of bias in the pipeline.

In [ ]:
threshold = pd.Series(val_preds_calibrated).quantile(0.90)
enrolled  = (val_preds_calibrated >= threshold).astype(int)

print("enrollment threshold:", round(threshold, 3))
print("overall enrollment rate:", round(enrolled.mean() * 100, 1), "%")

val_with_race = val[["race_clean", "hadm_id"]].copy()
val_with_race["enrolled"]   = enrolled
val_with_race["risk_score"] = val_preds_calibrated
val_with_race["y_true"]     = y_val.values

print("\nenrollment rate by race:")
print(val_with_race.groupby("race_clean")["enrolled"].mean().round(3).sort_values(ascending=False).to_string())

enrollment threshold: 0.37
overall enrollment rate: 10.9 %

enrollment rate by race:
race_clean
Hispanic/Latino           0.143
Black/African American    0.142
White                     0.107
Asian                     0.090
Other/Unknown             0.054


## Saving results

Saving the enrollment decisions alongside race, true labels, and risk scores. This is the baseline fairness measurement that all the mitigation methods in Week 2 will be compared against.

In [ ]:
val_with_race.to_parquet("/kaggle/working/stage3_results.parquet", index=False)

print("results saved")
print("\nsummary:")
print("  patients evaluated:", len(val_with_race))
print("  patients enrolled: ", enrolled.sum())
print("  enrollment rate:   ", round(enrolled.mean() * 100, 1), "%")
print("\nenrollment by race:")
print(val_with_race.groupby("race_clean")["enrolled"].agg(["sum", "mean"]).round(3).to_string())

results saved

summary:
  patients evaluated: 59905
  patients enrolled:  6541
  enrollment rate:    10.9 %

enrollment by race:
                         sum   mean
race_clean                         
Asian                    179  0.090
Black/African American  1258  0.142
Hispanic/Latino          443  0.143
Other/Unknown            254  0.054
White                   4407  0.107
